In [1]:
import glob, os
import numpy as np
import pandas as pd

root = os.path.dirname(glob.glob("/kaggle/input/**/aes_prepared.csv", recursive=True)[0])
print("Data folder:", root)
load = lambda name: pd.read_csv(os.path.join(root, name))

aes  = (load("aes_prepared.csv").merge(load("aes_features.csv"), on="essay_id")
                                .merge(load("aes_languagetool.csv"), on="essay_id"))
ell  = (load("ell_prepared.csv").merge(load("ell_features.csv"), on="text_id")
                                .merge(load("ell_languagetool.csv"), on="text_id"))
asap = (load("asap_prepared.csv").merge(load("asap_features.csv"), on="essay_id")
                                 .merge(load("asap_languagetool.csv"), on="essay_id"))

trait_cols = ["cohesion", "syntax", "vocabulary", "phraseology", "grammar", "conventions"]
NON_FEATURES = {"essay_id", "text_id", "text", "score", "fold", "word_count", "essay_set",
                "task_type", "domain1_score", "score_scaled", "domain2_score"} | set(trait_cols)

feature_cols = [c for c in aes.columns if c not in NON_FEATURES]
print(f"\n{len(feature_cols)} features")
print("Same features in all three tables:",
      all(c in ell.columns and c in asap.columns for c in feature_cols))

for name, df in [("AES", aes), ("ELL", ell), ("ASAP", asap)]:
    print(f"{name}: {df.shape[0]:,} rows, missing feature values: {int(df[feature_cols].isna().sum().sum())}")
print("\nFold sizes AES:", aes["fold"].value_counts().sort_index().tolist(),
      " ELL:", ell["fold"].value_counts().sort_index().tolist())

Data folder: /kaggle/input/datasets/faizanaliansari/essay-features

59 features
Same features in all three tables: True
AES: 17,307 rows, missing feature values: 0
ELL: 3,911 rows, missing feature values: 0
ASAP: 12,972 rows, missing feature values: 0

Fold sizes AES: [3462, 3462, 3461, 3461, 3461]  ELL: [783, 782, 782, 782, 782]


## Baselines

Before any deep learning we measure two simple models so every later improvement has something to beat:

- **Layer 0:** a straight line on essay length only.
- **Layer 1:** LightGBM on all 59 engineered features (surface, vocabulary, syntax, spelling, readability, LanguageTool).

Scoring uses the same 5 folds for every model. AES 2.0 is scored with QWK (predictions rounded to 1-6), and ELL with RMSE per trait compared with always predicting the average.

In [2]:
from sklearn.metrics import cohen_kappa_score
from sklearn.linear_model import LinearRegression
import lightgbm as lgb

def qwk(y_true, y_pred, lo=1, hi=6):
    return cohen_kappa_score(y_true, np.clip(np.rint(y_pred), lo, hi).astype(int), weights="quadratic")

params = dict(n_estimators=600, learning_rate=0.03, num_leaves=31, min_child_samples=30,
              subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
              random_state=42, verbose=-1)

# ---------------- AES 2.0 ----------------
y  = aes["score"].values
X0 = np.log1p(aes[["n_words"]].values)
oof0, oof1 = np.zeros(len(aes)), np.zeros(len(aes))
importance = pd.Series(0.0, index=feature_cols)

for f in range(5):
    tr, va = (aes["fold"] != f).values, (aes["fold"] == f).values
    oof0[va] = LinearRegression().fit(X0[tr], y[tr]).predict(X0[va])
    m1 = lgb.LGBMRegressor(**params).fit(aes.loc[tr, feature_cols], y[tr])
    oof1[va] = m1.predict(aes.loc[va, feature_cols])
    importance += pd.Series(m1.booster_.feature_importance("gain"), index=feature_cols)
    print(f"fold {f}: length-only QWK {qwk(y[va], oof0[va]):.3f} | LightGBM QWK {qwk(y[va], oof1[va]):.3f}")

print(f"\nAES 2.0 overall QWK  ->  Layer 0 (length only): {qwk(y, oof0):.3f}   Layer 1 (LightGBM): {qwk(y, oof1):.3f}")
print(f"AES 2.0 overall RMSE ->  Layer 0: {np.sqrt(np.mean((y - oof0) ** 2)):.3f}   Layer 1: {np.sqrt(np.mean((y - oof1) ** 2)):.3f}")

pred = np.clip(np.rint(oof1), 1, 6).astype(int)
print("\nLightGBM predictions vs truth (rows = true score, columns = predicted score):")
print(pd.crosstab(y, pred, rownames=["true"], colnames=["pred"]).to_string())

print("\nTop 15 features (share of total gain, %):")
print((importance / importance.sum() * 100).sort_values(ascending=False).head(15).round(1).to_string())

pd.DataFrame({"essay_id": aes["essay_id"], "fold": aes["fold"], "score": y,
              "oof_length": oof0, "oof_lgbm": oof1}).to_csv("/kaggle/working/oof_aes_baselines.csv", index=False)

# ---------------- ELL ----------------
Y = ell[trait_cols].values
oof_mean, oof_lgb = np.zeros_like(Y), np.zeros_like(Y)
for f in range(5):
    tr, va = (ell["fold"] != f).values, (ell["fold"] == f).values
    oof_mean[va] = Y[tr].mean(axis=0)
    for j, trait in enumerate(trait_cols):
        m = lgb.LGBMRegressor(**params).fit(ell.loc[tr, feature_cols], Y[tr, j])
        oof_lgb[va, j] = m.predict(ell.loc[va, feature_cols])

rmse = lambda a, b: np.sqrt(np.mean((a - b) ** 2, axis=0))
res = pd.DataFrame({"always the average": rmse(Y, oof_mean), "LightGBM": rmse(Y, oof_lgb)}, index=trait_cols)
res.loc["MEAN of 6 traits"] = res.mean()
print("\nELL RMSE per trait (lower is better):")
print(res.round(3).to_string())

fold 0: length-only QWK 0.639 | LightGBM QWK 0.744
fold 1: length-only QWK 0.631 | LightGBM QWK 0.757
fold 2: length-only QWK 0.623 | LightGBM QWK 0.739
fold 3: length-only QWK 0.630 | LightGBM QWK 0.750
fold 4: length-only QWK 0.635 | LightGBM QWK 0.752

AES 2.0 overall QWK  ->  Layer 0 (length only): 0.632   Layer 1 (LightGBM): 0.748
AES 2.0 overall RMSE ->  Layer 0: 0.738   Layer 1: 0.624

LightGBM predictions vs truth (rows = true score, columns = predicted score):
pred    1     2     3     4    5   6
true                                
1     187   873   175    16    1   0
2     115  3282  1256    67    3   0
3       7  1143  4267   839   23   1
4       0    23  1446  2268  187   2
5       0     0    26   549  389   6
6       0     0     0    23  123  10

Top 15 features (share of total gain, %):
n_chars                 42.7
n_words                 13.3
lt_nontypo_total         3.5
n_unique_words           2.7
lt_typos                 2.6
ttr                      2.6
pos_aux      

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

def tfidf_oof(df, targets, alpha=2.0):
    oof = np.zeros(targets.shape, dtype=float)
    for f in range(5):
        tr, va = (df["fold"] != f).values, (df["fold"] == f).values
        vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=200_000, sublinear_tf=True)
        Xtr = vec.fit_transform(df.loc[tr, "text"])
        Xva = vec.transform(df.loc[va, "text"])
        oof[va] = Ridge(alpha=alpha).fit(Xtr, targets[tr]).predict(Xva)
    return oof

# ---------------- AES 2.0 ----------------
oof_tf = tfidf_oof(aes, y)
blend  = 0.5 * oof1 + 0.5 * oof_tf
print("AES 2.0 QWK  ->  length only: %.3f | LightGBM: %.3f | TF-IDF Ridge: %.3f | 50/50 blend: %.3f"
      % (qwk(y, oof0), qwk(y, oof1), qwk(y, oof_tf), qwk(y, blend)))
print("AES 2.0 RMSE ->  LightGBM: %.3f | TF-IDF Ridge: %.3f | 50/50 blend: %.3f"
      % tuple(np.sqrt(np.mean((y - p) ** 2)) for p in (oof1, oof_tf, blend)))

# ---------------- ELL ----------------
oof_tf_ell = tfidf_oof(ell, Y)
blend_ell  = 0.5 * oof_lgb + 0.5 * oof_tf_ell
res2 = pd.DataFrame({"always the average": rmse(Y, oof_mean), "LightGBM": rmse(Y, oof_lgb),
                     "TF-IDF Ridge": rmse(Y, oof_tf_ell), "50/50 blend": rmse(Y, blend_ell)}, index=trait_cols)
res2.loc["MEAN of 6 traits"] = res2.mean()
print("\nELL RMSE per trait (lower is better):")
print(res2.round(3).to_string())

pd.DataFrame({"essay_id": aes["essay_id"], "oof_tfidf": oof_tf}).to_csv("/kaggle/working/oof_aes_tfidf.csv", index=False)

AES 2.0 QWK  ->  length only: 0.632 | LightGBM: 0.748 | TF-IDF Ridge: 0.716 | 50/50 blend: 0.754
AES 2.0 RMSE ->  LightGBM: 0.624 | TF-IDF Ridge: 0.632 | 50/50 blend: 0.594

ELL RMSE per trait (lower is better):
                  always the average  LightGBM  TF-IDF Ridge  50/50 blend
cohesion                       0.663     0.541         0.554        0.526
syntax                         0.644     0.505         0.529        0.493
vocabulary                     0.583     0.456         0.472        0.445
phraseology                    0.656     0.519         0.529        0.502
grammar                        0.700     0.557         0.572        0.538
conventions                    0.671     0.491         0.561        0.494
MEAN of 6 traits               0.653     0.512         0.536        0.500
